# BeetleCast 07 — Complete ground-truth year audit

This notebook provides a reproducible answer to:

> **Which years have actual supplied bark-beetle damage polygons?**

It audits the original label file rather than inferring truth from Sentinel-2,
AlphaEarth, LULC, or model predictions.

The notebook:

1. strictly locates the original bark-beetle locations file and the F3 bark-beetle AOI;
2. inspects all label attributes for year values;
3. extracts the survey/source year from `tile_name`;
4. checks **all years from 2000 through 2025**, including every year before 2022;
5. clips the labels to the F3 AOI;
6. exports a complete year-by-year summary and audit report.

Expected conclusion for the supplied dataset:

- no supplied polygon labels exist before 2022;
- actual polygon labels exist for **2022 and 2023**;
- no supplied polygon labels exist for **2024 or 2025**.


In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

cwd = Path.cwd()

if (cwd / "hackathon_data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "hackathon_data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "ground_truth_audit"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


PROJECT_ROOT: /Users/hemat/Desktop/hackathon-demo
OUTPUT_ROOT: /Users/hemat/Desktop/hackathon-demo/outputs/ground_truth_audit


## 1. Locate the original label and AOI files

In [2]:
# Optional explicit overrides.
# Leave as None to use strict automatic discovery.
LABEL_PATH_OVERRIDE = None
AOI_PATH_OVERRIDE = None

def normalised_name(path):
    return re.sub(r"[^a-z0-9]+", "", path.name.lower())

def is_bark_beetle_label(path):
    name = normalised_name(path)

    return (
        ("barkbeetle" in name or "barkbettle" in name)
        and "location" in name
        and "aoi" not in name
    )

def is_f3_bark_beetle_aoi(path):
    name = normalised_name(path)

    return (
        "f3" in name
        and ("barkbeetle" in name or "barkbettle" in name)
        and "aoi" in name
    )

search_roots = [
    PROJECT_ROOT,
    PROJECT_ROOT / "hackathon_data",
    PROJECT_ROOT / "hackathon_demo_data",
]

vector_paths = []

for root in search_roots:
    if root.exists():
        vector_paths.extend(root.rglob("*.geojson"))
        vector_paths.extend(root.rglob("*.gpkg"))
        vector_paths.extend(root.rglob("*.shp"))

vector_paths = sorted(
    set(path.resolve() for path in vector_paths)
)

if LABEL_PATH_OVERRIDE is not None:
    LABEL_PATH = Path(LABEL_PATH_OVERRIDE).expanduser().resolve()
else:
    label_candidates = [
        path for path in vector_paths
        if is_bark_beetle_label(path)
    ]

    assert label_candidates, (
        "Could not find a bark-beetle locations file. "
        "Expected a name similar to "
        "F3_bark_beetle_locations.geojson."
    )

    # Prefer raw GeoJSON over processed outputs.
    label_candidates = sorted(
        label_candidates,
        key=lambda path: (
            "processed" in str(path).lower(),
            path.suffix.lower() != ".geojson",
            len(str(path)),
        ),
    )

    LABEL_PATH = label_candidates[0]

if AOI_PATH_OVERRIDE is not None:
    AOI_PATH = Path(AOI_PATH_OVERRIDE).expanduser().resolve()
else:
    aoi_candidates = [
        path for path in vector_paths
        if is_f3_bark_beetle_aoi(path)
    ]

    assert aoi_candidates, (
        "Could not find the F3 bark-beetle AOI. "
        "Expected a name similar to "
        "F3_Germany_BarkBeetle_aoi.geojson."
    )

    # Prefer GeoJSON and the shortest canonical-looking path.
    aoi_candidates = sorted(
        aoi_candidates,
        key=lambda path: (
            path.suffix.lower() != ".geojson",
            "(1)" in path.name,
            len(str(path)),
        ),
    )

    AOI_PATH = aoi_candidates[0]

assert LABEL_PATH.exists(), f"Label file does not exist: {LABEL_PATH}"
assert AOI_PATH.exists(), f"AOI file does not exist: {AOI_PATH}"

assert is_bark_beetle_label(LABEL_PATH), (
    "Selected label file is not a bark-beetle locations file: "
    f"{LABEL_PATH}"
)

assert is_f3_bark_beetle_aoi(AOI_PATH), (
    "Selected AOI is not the F3 bark-beetle AOI: "
    f"{AOI_PATH}"
)

print("STRICT FILE SELECTION")
print("---------------------")
print("Label file:", LABEL_PATH)
print("AOI file:", AOI_PATH)

print()
print("All matching bark-beetle label candidates:")
for path in [
    path for path in vector_paths
    if is_bark_beetle_label(path)
]:
    print("-", path)

print()
print("All matching F3 bark-beetle AOI candidates:")
for path in [
    path for path in vector_paths
    if is_f3_bark_beetle_aoi(path)
]:
    print("-", path)


STRICT FILE SELECTION
---------------------
Label file: /Users/hemat/Desktop/hackathon-demo/hackathon_data/raw/F3_bark_beetle_locations.geojson
AOI file: /Users/hemat/Desktop/hackathon-demo/hackathon_data/clean/F3_bark_beetle_labels_in_aoi.geojson

All matching bark-beetle label candidates:
- /Users/hemat/Desktop/hackathon-demo/hackathon_data/raw/F3_bark_beetle_locations.geojson

All matching F3 bark-beetle AOI candidates:
- /Users/hemat/Desktop/hackathon-demo/hackathon_data/clean/F3_bark_beetle_labels_in_aoi.geojson
- /Users/hemat/Desktop/hackathon-demo/hackathon_data/clean/F3_bark_beetle_labels_in_aoi.gpkg


## 2. Inspect the raw files

In [3]:
labels_raw = gpd.read_file(LABEL_PATH)
aoi = gpd.read_file(AOI_PATH)

print("RAW LABEL FILE")
print("--------------")
print("Rows:", len(labels_raw))
print("CRS:", labels_raw.crs)
print("Geometry types:")
print(labels_raw.geom_type.value_counts())
print("Columns:", labels_raw.columns.tolist())

print()
print("AOI FILE")
print("--------")
print("Rows:", len(aoi))
print("CRS:", aoi.crs)
print("Geometry types:")
print(aoi.geom_type.value_counts())
print("Columns:", aoi.columns.tolist())

assert "geometry" in labels_raw.columns
assert not labels_raw.empty
assert not aoi.empty


RAW LABEL FILE
--------------
Rows: 14358
CRS: EPSG:4326
Geometry types:
MultiPolygon    14358
Name: count, dtype: int64
Columns: ['fid', 'tile_name', 'area_ha', 'geometry']

AOI FILE
--------
Rows: 3014
CRS: EPSG:4326
Geometry types:
Polygon         3012
MultiPolygon       2
Name: count, dtype: int64
Columns: ['fid', 'tile_name', 'label_year', 'area_ha', 'clipped_area_ha', 'geometry']


In [5]:
# Safety checks against accidentally selecting another challenge AOI.
selected_aoi_name = normalised_name(AOI_PATH)

assert "vineyard" not in selected_aoi_name
assert "chablis" not in selected_aoi_name
assert "f3" in selected_aoi_name
assert "aoi" in selected_aoi_name
assert (
    "barkbeetle" in selected_aoi_name
    or "barkbettle" in selected_aoi_name
)

assert len(aoi) == 1, (
    "Expected the F3 AOI file to contain one boundary feature."
)

print("AOI safety check passed.")
print("Confirmed F3 bark-beetle AOI:", AOI_PATH.name)


AssertionError: Expected the F3 AOI file to contain one boundary feature.

## 3. Search every non-geometry attribute for four-digit years

This prevents the audit from relying only on one assumed column.


In [ ]:
year_occurrences = []

for column in labels_raw.columns:
    if column == "geometry":
        continue

    values = labels_raw[column].astype(str)

    extracted = values.str.extractall(r"(?<!\d)(20\d{2})(?!\d)")

    if extracted.empty:
        continue

    counts = extracted[0].value_counts().sort_index()

    for year_text, count in counts.items():
        year_occurrences.append({
            "column": column,
            "year": int(year_text),
            "occurrences": int(count),
        })

attribute_years = pd.DataFrame(year_occurrences)

if attribute_years.empty:
    print("No four-digit years found in any attribute.")
else:
    display(attribute_years)

all_attribute_years = sorted(
    attribute_years["year"].unique().tolist()
) if not attribute_years.empty else []

print("All years found anywhere in label attributes:", all_attribute_years)


## 4. Extract the label year from `tile_name`

The supplied year appears in names such as:

```text
..._rp_2022_...
..._rp_2023_...
```

A strict pattern is used so unrelated numbers in the filename are not mistaken for years.


In [ ]:
assert "tile_name" in labels_raw.columns, (
    "The supplied label file does not contain tile_name."
)

strict_year = labels_raw["tile_name"].astype(str).str.extract(
    r"_rp_(20\d{2})_",
    expand=False,
)

labels_raw = labels_raw.copy()
labels_raw["label_year"] = pd.to_numeric(
    strict_year,
    errors="coerce",
).astype("Int64")

unmatched = int(labels_raw["label_year"].isna().sum())

print("Unmatched tile_name rows:", unmatched)
assert unmatched == 0, (
    "Some labels do not contain the expected _rp_YEAR_ pattern."
)

raw_year_summary = (
    labels_raw.groupby("label_year", dropna=False)
    .agg(
        polygon_count=("geometry", "size"),
        labelled_area_ha=("area_ha", "sum")
        if "area_ha" in labels_raw.columns
        else ("geometry", "size"),
    )
    .reset_index()
)

display(raw_year_summary)

raw_years = sorted(
    labels_raw["label_year"].dropna().astype(int).unique().tolist()
)

print("Unique strict label years:", raw_years)


## 5. Explicitly test every year from 2000 through 2025

In [ ]:
AUDIT_START_YEAR = 2000
AUDIT_END_YEAR = 2025

audit_rows = []

for year in range(AUDIT_START_YEAR, AUDIT_END_YEAR + 1):
    count = int(
        (labels_raw["label_year"] == year).sum()
    )

    if count > 0:
        status = "labelled ground truth"
    elif year < 2022:
        status = "no supplied polygon ground truth before 2022"
    else:
        status = "no supplied polygon ground truth"

    audit_rows.append({
        "year": year,
        "raw_ground_truth_polygon_count": count,
        "has_supplied_ground_truth": count > 0,
        "status": status,
    })

year_audit = pd.DataFrame(audit_rows)
display(year_audit)

pre_2022_count = int(
    year_audit.loc[
        year_audit["year"] < 2022,
        "raw_ground_truth_polygon_count",
    ].sum()
)

print("Total supplied polygons before 2022:", pre_2022_count)
print(
    "Years before 2022 with any supplied polygons:",
    year_audit.loc[
        (year_audit["year"] < 2022)
        & year_audit["has_supplied_ground_truth"],
        "year",
    ].tolist(),
)

assert pre_2022_count == 0, (
    "Unexpected supplied labels were found before 2022."
)

assert int(
    year_audit.loc[
        year_audit["year"] == 2022,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) > 0

assert int(
    year_audit.loc[
        year_audit["year"] == 2023,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) > 0

assert int(
    year_audit.loc[
        year_audit["year"] == 2024,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) == 0

assert int(
    year_audit.loc[
        year_audit["year"] == 2025,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) == 0

print("\nAssertions passed:")
print("- no supplied polygon labels exist from 2000–2021")
print("- 2022 labels exist")
print("- 2023 labels exist")
print("- 2024 labels do not exist")
print("- 2025 labels do not exist")


## 6. Clip the ground truth to the F3 AOI

In [ ]:
if labels_raw.crs is None:
    raise ValueError("The label file has no CRS.")

if aoi.crs is None:
    raise ValueError("The AOI file has no CRS.")

aoi_in_label_crs = aoi.to_crs(labels_raw.crs)

# Repair invalid geometries conservatively before clipping.
labels_valid = labels_raw[
    labels_raw.geometry.notna()
    & ~labels_raw.geometry.is_empty
].copy()

labels_valid["geometry"] = labels_valid.geometry.make_valid()

aoi_geometry = aoi_in_label_crs.geometry.union_all()

labels_in_aoi = labels_valid[
    labels_valid.geometry.intersects(aoi_geometry)
].copy()

labels_in_aoi["geometry"] = labels_in_aoi.geometry.intersection(
    aoi_geometry
)

labels_in_aoi = labels_in_aoi[
    labels_in_aoi.geometry.notna()
    & ~labels_in_aoi.geometry.is_empty
].copy()

# Calculate area in a suitable projected CRS.
projected_crs = aoi_in_label_crs.estimate_utm_crs()
labels_projected = labels_in_aoi.to_crs(projected_crs)
labels_in_aoi["clipped_area_ha"] = (
    labels_projected.geometry.area / 10_000
)

aoi_year_summary = (
    labels_in_aoi.groupby("label_year")
    .agg(
        polygon_count=("geometry", "size"),
        clipped_area_ha=("clipped_area_ha", "sum"),
    )
    .reset_index()
)

display(aoi_year_summary)

aoi_years = sorted(
    labels_in_aoi["label_year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print("Ground-truth years inside F3 AOI:", aoi_years)

assert aoi_years == [2022, 2023], (
    f"Unexpected AOI label years: {aoi_years}"
)


## 7. Visual evidence

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    constrained_layout=True,
)

raw_plot = raw_year_summary.copy()
raw_plot["label_year"] = raw_plot["label_year"].astype(str)

axes[0].bar(
    raw_plot["label_year"],
    raw_plot["polygon_count"],
)
axes[0].set_title("Raw supplied damage polygons by year")
axes[0].set_xlabel("Label year")
axes[0].set_ylabel("Polygon count")

aoi_plot = aoi_year_summary.copy()
aoi_plot["label_year"] = aoi_plot["label_year"].astype(str)

axes[1].bar(
    aoi_plot["label_year"],
    aoi_plot["polygon_count"],
)
axes[1].set_title("Damage polygons inside the F3 AOI")
axes[1].set_xlabel("Label year")
axes[1].set_ylabel("Polygon count")

figure_path = OUTPUT_ROOT / "ground_truth_year_counts.png"
plt.savefig(
    figure_path,
    dpi=220,
    bbox_inches="tight",
)
plt.show()

print("Saved:", figure_path)


## 8. Export the reproducible audit outputs

In [ ]:
raw_summary_path = OUTPUT_ROOT / "raw_ground_truth_year_summary.csv"
aoi_summary_path = OUTPUT_ROOT / "aoi_ground_truth_year_summary.csv"
audit_path = OUTPUT_ROOT / "ground_truth_availability_2000_2025.csv"
attribute_years_path = OUTPUT_ROOT / "all_attribute_year_occurrences.csv"
clean_labels_path = OUTPUT_ROOT / "F3_ground_truth_2022_2023.gpkg"

raw_year_summary.to_csv(raw_summary_path, index=False)
aoi_year_summary.to_csv(aoi_summary_path, index=False)
year_audit.to_csv(audit_path, index=False)
attribute_years.to_csv(attribute_years_path, index=False)

labels_in_aoi.to_file(
    clean_labels_path,
    layer="bark_beetle_ground_truth",
    driver="GPKG",
)

report = {
    "source_label_file": str(LABEL_PATH),
    "source_aoi_file": str(AOI_PATH),
    "raw_polygon_count": int(len(labels_raw)),
    "raw_label_years": raw_years,
    "aoi_polygon_count": int(len(labels_in_aoi)),
    "aoi_label_years": aoi_years,
    "has_pre_2022_ground_truth": any(year < 2022 for year in raw_years),
    "has_2022_ground_truth": 2022 in raw_years,
    "has_2023_ground_truth": 2023 in raw_years,
    "has_2024_ground_truth": 2024 in raw_years,
    "has_2025_ground_truth": 2025 in raw_years,
    "conclusion": (
        "The supplied polygon ground truth covers 2022 and 2023 only; "
        "no supplied polygon labels exist before 2022. "
        "No supplied polygon ground truth is available for 2024 or 2025."
    ),
}

report_path = OUTPUT_ROOT / "ground_truth_audit_report.json"
with open(report_path, "w") as handle:
    json.dump(report, handle, indent=2)

print("Saved:")
for path in [
    raw_summary_path,
    aoi_summary_path,
    audit_path,
    attribute_years_path,
    clean_labels_path,
    report_path,
]:
    print("-", path)


## 9. Final conclusion

In [ ]:
print("GROUND-TRUTH AUDIT COMPLETE")
print("---------------------------")
print(
    f"Raw supplied polygons: {len(labels_raw):,}"
)
print(
    "Raw supplied label years:",
    raw_years,
)
print(
    f"Polygons inside F3 AOI: {len(labels_in_aoi):,}"
)
print(
    "Label years inside F3 AOI:",
    aoi_years,
)
print()
print("Ground-truth availability:")
print("- 2000–2021: 0 supplied polygons")
print(
    "- 2022:",
    f"{int((labels_raw['label_year'] == 2022).sum()):,}",
    "supplied polygons",
)
print(
    "- 2023:",
    f"{int((labels_raw['label_year'] == 2023).sum()):,}",
    "supplied polygons",
)
print("- 2024: 0 supplied polygons")
print("- 2025: 0 supplied polygons")
print()
print(
    "Conclusion: the supplied bark-beetle polygon ground truth "
    "exists only for 2022 and 2023."
)
print(
    "There are no supplied polygon labels before 2022, "
    "or for 2024 and 2025."
)
print(
    "Risk maps for unlabelled years must be described as "
    "predictions or inspection-priority maps."
)

print()
print("Files used:")
print("- Labels:", LABEL_PATH)
print("- F3 bark-beetle AOI:", AOI_PATH)
